# SchemeSaathi — Hindi translation on Google Colab (open-source model)

Translates the imported schemes' English text into Hindi with **IndicTrans2** (AI4Bharat, MIT licence),
running free on a Colab GPU. No paid API.

**In:** `work.zip` — your `data/translations/work/` folder (`manifest.json`, `todo/`, `done/`).
**Out:** `done.zip` — `done/NNNN.json` files in the format `npm run data:translate:merge` expects.

### On your computer, first
```bash
cd server && npm run data:translate:export      # only if data/translations/work/ doesn't exist yet
cd ../data/translations && zip -r work.zip work  # → data/translations/work.zip
```

### In Colab
1. *Runtime → Change runtime type → **T4 GPU***.
2. One time: open <https://huggingface.co/ai4bharat/indictrans2-en-indic-1B>, log in and click **Agree** (access is instant).
   Create a *read* token at <https://huggingface.co/settings/tokens> and add it to Colab secrets (🔑 on the left) as `HF_TOKEN`
   with notebook access on.
3. *Runtime → Run all*. Upload `work.zip` when asked. At the end `done.zip` downloads.

### Back on your computer
```bash
cd data/translations/work && unzip -o ~/Downloads/done.zip   # → data/translations/work/done/NNNN.json
cd ../../../server
npm run data:translate:merge     # validates + writes data/translations/hi.json
npm run data:build               # fills Hindi into data/processed/catalogue.json
npm run db:import                # loads it into PostgreSQL (docker compose up -d db first)
```

Chunks already in `done/` (e.g. earlier translations) are kept, not re-translated.
The run takes roughly 1–3 hours on a T4. If Colab may disconnect, set `SAVE_TO_DRIVE = True`. Progress is then kept
in Google Drive, and *Run all* again continues where it stopped.

## 1. Settings

In [ ]:
MODEL = "ai4bharat/indictrans2-en-indic-1B"   # smaller/faster, slightly lower quality: "ai4bharat/indictrans2-en-indic-dist-200M"
BATCH_SIZE = 32        # sentences per GPU batch; lower to 16 if you get "CUDA out of memory"
NUM_BEAMS = 5          # 1 = ~3x faster, a little lower quality
CHUNKS_PER_ROUND = 10  # done/ files are written after every round
SAVE_TO_DRIVE = False  # True: keep the work folder in Google Drive so a disconnect loses nothing

## 2. Install and check the GPU

In [ ]:
# IndicTransToolkit and IndicTrans2's model code need transformers 4.x (Colab now ships 5.x)
%pip install -q indictranstoolkit "transformers>=4.51,<5"
import sys, importlib.metadata as md
if "transformers" in sys.modules and sys.modules["transformers"].__version__.startswith("5"):
    raise SystemExit("transformers 5 is still loaded: Runtime → Restart session, then Run all again")
print("transformers", md.version("transformers"))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — Runtime → Change runtime type → T4 GPU")

## 3. Upload `work.zip`

In [ ]:
import zipfile, glob, os, json, re, shutil
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/schemesaathi-translation")
else:
    BASE = Path("/content/schemesaathi-translation")
BASE.mkdir(parents=True, exist_ok=True)

def find_work():
    hits = sorted(BASE.glob("**/manifest.json"), key=lambda p: len(p.parts))
    return hits[0].parent if hits else None

WORK = find_work()
if WORK is None:
    from google.colab import files
    print("Choose work.zip …")
    uploaded = files.upload()
    name = next(n for n in uploaded if n.endswith(".zip"))
    with zipfile.ZipFile(name) as z:
        z.extractall(BASE / "unzipped")
    os.remove(name)
    WORK = find_work()
    if WORK is None:
        raise FileNotFoundError("work.zip has no manifest.json — zip the data/translations/work folder")
else:
    print("Using the work folder already in", BASE, "(delete it to start over with a new work.zip)")

TODO, DONE = WORK / "todo", WORK / "done"
DONE.mkdir(exist_ok=True)
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
chunks = sorted(p.name for p in TODO.glob("*.json"))
print(f"{len(chunks)} chunks, {len(manifest)} strings, {len(list(DONE.glob('*.json')))} done files")

## 4. Checks (same rules as `npm run data:translate:merge`)

In [ ]:
DEVANAGARI = re.compile("[\u0900-\u097F]")

def check(en, hi):
    if not isinstance(hi, str) or not hi.strip():
        return "empty"
    if not DEVANAGARI.search(hi) and re.search("[a-z]{3,}", en):
        return "no Devanagari"
    if len(en) > 200 and len(hi) < len(en) * 0.4:
        return "too short (truncated?)"
    if len(en.strip().split("\n")) != len(hi.strip().split("\n")):
        return "line count differs"
    return None

def read_done(name):
    try:
        return json.loads((DONE / name).read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def missing_items(name):
    todo = json.loads((TODO / name).read_text(encoding="utf-8"))
    done = read_done(name)
    for t in todo:
        if manifest.get(t["id"]) != t["en"]:
            raise RuntimeError(f"{name}: todo/ and manifest.json come from different exports — re-run data:translate:export and zip again")
    return [t for t in todo if check(t["en"], done.get(t["id"]))]

pending = [n for n in chunks if missing_items(n)]
left = sum(len(t["en"]) for n in pending for t in missing_items(n))
print(f"{len(chunks) - len(pending)} chunks complete, {len(pending)} to translate ({left:,} characters)")

## 5. Splitting text into sentences

IndicTrans2 translates one sentence at a time (up to ~256 tokens), so each string is split into lines, then sentences.
Bullet and step markers (`•`, `1.`, `(ii)`) are kept as they are, and the Hindi is put back together line by line.
This keeps the "one line in → one line out" rule of `GUIDE.md`.

In [ ]:
MARKER = re.compile(r"^\s*(?:[•●▪◦►✓*–-]+(?=\s)|\(?\d{1,2}[.)](?=\s)|\(?[ivxIVX]{1,4}\)|\(?[a-hA-H]\))?\s*")
ABBREV = {"rs", "dr", "mr", "mrs", "ms", "no", "nos", "st", "vs", "sr", "jr", "prof", "smt", "shri", "sh", "govt",
          "dept", "e.g", "i.e", "viz", "approx", "ltd", "pvt", "co", "sec", "art", "km", "kg", "max", "min", "u.s"}
MAX_WORDS = 60

def keep_as_is(text):
    # nothing to translate: numbers, codes, a URL or an email address
    return not re.search("[A-Za-z]{2,}", text) or re.fullmatch(r"\S+@\S+|(https?://|www\.)\S+", text) is not None

def split_long(sentence):
    words = sentence.split()
    if len(words) <= MAX_WORDS:
        return [sentence]
    parts, cur = [], []
    for piece in re.split(r"(?<=[;:,])\s+", sentence):
        if cur and len(" ".join(cur + [piece]).split()) > MAX_WORDS:
            parts.append(" ".join(cur)); cur = []
        cur.append(piece)
    parts.append(" ".join(cur))
    out = []
    for p in parts:   # a single comma-free run that is still too long: cut by words
        w = p.split()
        out += [" ".join(w[i:i + MAX_WORDS]) for i in range(0, len(w), MAX_WORDS)]
    return out

def sentences(text):
    raw = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"“(])", text)
    merged = []
    for s in raw:
        last = merged[-1].rsplit(None, 1)[-1].rstrip(".").lower() if merged else ""
        if merged and (last in ABBREV or len(last) == 1):   # "Rs. 5000", "Dr. Ambedkar", initials
            merged[-1] += " " + s
        else:
            merged.append(s)
    return [p for s in merged for p in split_long(s)]

def plan(en):
    """Returns the string's lines as [(marker, [sentences] or None if kept, ellipsis)]."""
    lines = []
    for line in en.strip().split("\n"):
        marker = MARKER.match(line).group(0)
        body = line[len(marker):].strip()
        ellipsis = "…" if re.search(r"(…|\.\.\.)$", body) else ""
        if ellipsis:
            body = re.sub(r"(…|\.\.\.)$", "", body).rstrip()
        lines.append((marker, None if keep_as_is(body) else sentences(body), body, ellipsis))
    return lines

def assemble(en, hindi_of):
    out = []
    for marker, sents, body, ellipsis in plan(en):
        text = body if sents is None else " ".join(hindi_of[s] for s in sents)
        out.append((marker + text + ellipsis).strip())
    return "\n".join(out)

# quick look
for s in ["• Aadhaar Card\n• Income Certificate", "Loan upto Rs. 1,50,000 for ITI students. Apply at www.nsp.gov.in."]:
    print(plan(s))

## 6. Load IndicTrans2

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from huggingface_hub import login
try:
    from IndicTransToolkit import IndicProcessor
except ImportError:
    from IndicTransToolkit.IndicTransToolkit import IndicProcessor

try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("No HF_TOKEN secret found (", e, ") — log in below")
    from huggingface_hub import notebook_login
    notebook_login()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL, trust_remote_code=True, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE).eval()
ip = IndicProcessor(inference=True)
USE_CACHE = True

def translate_batch(batch):
    global USE_CACHE
    pre = ip.preprocess_batch(batch, src_lang="eng_Latn", tgt_lang="hin_Deva")
    enc = tokenizer(pre, padding="longest", truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        try:
            out = model.generate(**enc, num_beams=NUM_BEAMS, max_length=256, use_cache=USE_CACHE)
        except Exception as e:   # some transformers versions break IndicTrans2's KV cache
            if not USE_CACHE:
                raise
            print("generate() failed with use_cache=True, retrying without:", type(e).__name__)
            USE_CACHE = False
            out = model.generate(**enc, num_beams=NUM_BEAMS, max_length=256, use_cache=False)
    dec = tokenizer.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return ip.postprocess_batch(dec, lang="hin_Deva")

def translate_all(segments):
    """Unique sentences → Hindi, longest first so batches have similar lengths."""
    order = sorted(set(segments), key=len, reverse=True)
    result = {}
    for i in range(0, len(order), BATCH_SIZE):
        batch = order[i:i + BATCH_SIZE]
        result.update(zip(batch, translate_batch(batch)))
    return result

print(translate_batch(["Skill Loan Scheme", "Ministry Of Social Justice and Empowerment",
                       "Annual family income should not exceed Rs. 2,50,000."]))

## 7. Translate

In [ ]:
import time
from tqdm.auto import tqdm

def write_done(name, data):
    tmp = DONE / (name + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=0), encoding="utf-8")
    tmp.replace(DONE / name)   # atomic: a disconnect never leaves a broken file

problems = {}
pending = [n for n in chunks if missing_items(n)]
start = time.time()
for r in tqdm(range(0, len(pending), CHUNKS_PER_ROUND), desc="rounds"):
    names = pending[r:r + CHUNKS_PER_ROUND]
    items = {n: missing_items(n) for n in names}
    segs = [s for its in items.values() for t in its for _, sents, _, _ in plan(t["en"]) if sents for s in sents]
    hindi_of = translate_all(segs)
    for n, its in items.items():
        done = read_done(n)
        bad = []
        for t in its:
            hi = assemble(t["en"], hindi_of)
            reason = check(t["en"], hi)
            if reason:
                bad.append(f"{t['id']} ({reason})")
            else:
                done[t["id"]] = hi
        write_done(n, done)
        if bad:
            problems[n] = bad
    mins = (time.time() - start) / 60
    print(f"{min(r + CHUNKS_PER_ROUND, len(pending))}/{len(pending)} chunks · {mins:.0f} min")

print(f"Strings that failed the checks: {sum(map(len, problems.values()))}")
for n, bad in list(problems.items())[:20]:
    print(" ", n, bad[:5])

## 8. Spot-check some translations

Read a few before downloading. Machine translation needs a native speaker's review before real use (see `docs/IMPROVEMENTS.md`).

In [ ]:
import random
sample = random.sample([(n, k) for n in chunks for k in read_done(n)], 8)
for n, k in sample:
    print("EN:", manifest[k][:250]); print("HI:", read_done(n)[k][:250]); print()

## 9. Download `done.zip`

Then on your computer:
```bash
cd data/translations/work && unzip -o ~/Downloads/done.zip
cd ../../../server && npm run data:translate:merge && npm run data:build && npm run db:import
```

In [ ]:
out = Path("/content/done.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(DONE.glob("*.json")):
        z.write(p, f"done/{p.name}")
complete = sum(1 for n in chunks if not missing_items(n))
print(f"done.zip: {len(list(DONE.glob('*.json')))} files, {complete}/{len(chunks)} chunks complete")
from google.colab import files
files.download(str(out))